# Feature Engineering Store
In this notebook, we apply time-safe aggregations to generate >100 quantitative features. We ensure no future leakages occur (all rolling statistics use `shift(1)`) and compile a complete catalog documenting the feature store.

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import sys

sys.path.append('../')
from src.feature_engineering import generate_features

ROOT = Path.cwd().parent
CSV = ROOT / 'csv_files'
DATA = ROOT / 'data'

## Run Feature Store Engine
We load the clean merged dataset and call our feature store engine.

In [2]:
df = pd.read_csv(CSV / 'historical_with_sentiment_clean.csv')
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])
df = df.sort_values('timestamp_utc').reset_index(drop=True)

df_feat = generate_features(df)
print(f"Feature matrix dimensions: {df_feat.shape}")

# Filter out nan rows where features haven't accumulated yet
print(f"Initial rows: {len(df_feat)}")
df_feat = df_feat[df_feat['acc_wr20'].notna()].copy()
print(f"Remaining rows after rolling warm-up filter: {len(df_feat)}")

Feature matrix dimensions: (211082, 113)
Initial rows: 211082


Remaining rows after rolling warm-up filter: 211076


## Compile Feature Catalog
We build a feature catalog of all the features in the model and candidate pools.

In [3]:
all_cols = df_feat.columns.tolist()
feat_rows = []

base_cols = ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp', 'timestamp_ist', 'timestamp_utc', 'date_utc', 'fg_timestamp', 'fg_value', 'fg_classification', 'Closed PnL_w', 'Size USD_w', 'Fee_w', 'y', 'date']

for c in all_cols:
    if c in base_cols:
        continue
    if 'acc_' in c:
        cat = 'Account Rolling Metrics'
        desc = 'Rolling window metric computed on historical account data with a shift(1) lag.'
    elif 'coin_' in c:
        cat = 'Coin Rolling Metrics'
        desc = 'Rolling window metric computed on historical coin performance data with a shift(1) lag.'
    elif 'fg_' in c:
        cat = 'Sentiment Dynamics'
        desc = 'Lagged values, moving averages, or changes in the Fear & Greed index.'
    elif 'sent_' in c or 'size_x' in c or 'fee_x' in c:
        cat = 'Cross-Feature Interactions'
        desc = 'Interaction cross-products (e.g. sentiment value multiplied by trader skill).'
    else:
        cat = 'Microstructure & Context'
        desc = 'Derived time feature or logarithmic base trade features.'
    feat_rows.append({'feature': c, 'category': cat, 'description': desc})

catalog_df = pd.DataFrame(feat_rows)
print(f"Cataloged {len(catalog_df)} engineered features.")
catalog_df.to_csv(CSV / 'feature_catalog_150.csv', index=False)
catalog_df.to_csv(DATA / 'feature_catalog_150.csv', index=False)
catalog_df.head(15)

Cataloged 86 engineered features.


,feature,category,description
0,size_usd_log,Microstructure & Context,Derived time feature or logarithmic base trade...
1,fee_abs_log,Microstructure & Context,Derived time feature or logarithmic base trade...
2,exec_price_log,Microstructure & Context,Derived time feature or logarithmic base trade...
3,sent_regime_ord,Cross-Feature Interactions,Interaction cross-products (e.g. sentiment val...
4,hour,Microstructure & Context,Derived time feature or logarithmic base trade...
5,dow,Microstructure & Context,Derived time feature or logarithmic base trade...
6,month,Microstructure & Context,Derived time feature or logarithmic base trade...
7,acc_wr5,Account Rolling Metrics,Rolling window metric computed on historical a...
8,acc_wr10,Account Rolling Metrics,Rolling window metric computed on historical a...
9,acc_wr20,Account Rolling Metrics,Rolling window metric computed on historical a...


## Save Feature Matrix
We save the final features dataframe for training models.

In [4]:
df_feat.to_csv(DATA / 'historical_with_features.csv', index=False)
print("Feature table exported successfully to data/historical_with_features.csv")

Feature table exported successfully to data/historical_with_features.csv
